In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from getdist import plots

plt.style.use("/home/guerrini/matplotlib_config/paper.mplstyle")

sns.set_palette("husl")

g = plots.get_subplot_plotter(width_inch=30)
g.settings.axes_fontsize = 60
g.settings.axes_labelsize = 60
g.settings.alpha_filled_add = 0.7
g.settings.legend_fontsize = 60

%matplotlib inline

# SPECIFY DATA DIRECTORY AND DESIRED CHAINS TO ANALYSE
root_dir = "/n09data/guerrini/glass_mock_chains/"

roots = [f"glass_mock_v0_{i:05d}" for i in range(1, 3)] + [
    f"glass_mock_v0_{i:05d}_cell" for i in range(1, 3)
]

print(roots)

In [ ]:
# MAKE PARAMNAMES FILE

for root in roots:
    if "_cell" in root:
        path_root = root.replace("_cell", "")
    else:
        path_root = root
    with open(
        root_dir + "{}/{}/samples_{}.txt".format("/" + path_root, path_root, root), "r"
    ) as file:
        params = file.readline()[1:].split("\t")[:-4]
        file.close()

    with open(
        root_dir
        + "{}/{}/getdist_{}.paramnames".format("/" + path_root, path_root, root),
        "w",
    ) as file:
        for i in range(len(params)):
            if len(params[i].split("--")) > 1:
                file.write(params[i].split("--")[1] + "\n")
            else:
                file.write(params[i].split("--")[0] + "\n")
        file.close()

In [ ]:
# READ CHAIN

chains = []

for root in roots:
    if "_cell" in root:
        path_root = root.replace("_cell", "")
    else:
        path_root = root
    samples = np.loadtxt(
        root_dir + "{}/{}/samples_{}.txt".format(path_root, path_root, root)
    )
    print(len(samples))
    if "nautilus" in root:
        samples = np.column_stack(
            (np.exp(samples[:, -3]), samples[:, -1] - samples[:, -2], samples[:, 0:-3])
        )
    else:
        samples = np.column_stack((samples[:, -1], samples[:, -3], samples[:, 0:-4]))
    np.savetxt(
        root_dir + "{}/{}/getdist_{}.txt".format(path_root, path_root, root), samples
    )

    chain = g.samples_for_root(
        root_dir + "{}/{}/getdist_{}".format(path_root, path_root, root),
        cache=False,
        settings={"ignore_rows": 0, "smooth_scale_2D": 0.5, "smooth_scale_1D": 0.5},
    )

    chains.append(chain)

In [ ]:
name_list = [
    "OMEGA_M",
    "ombh2",
    "h0",
    "n_s",
    "SIGMA_8",
    "s_8_input",
    "logt_agn",
    "a",
    "m1",
    "bias_1",
]
label_list = [
    r"\Omega_m",
    r"\omega_b h^2",
    "h_0",
    "n_s",
    r"\sigma_8",
    "S_8",
    "log T_{AGN}",
    "A_{IA}",
    "m_1",
    r"\Delta z_1",
]

for chain in chains:
    param_names = chain.getParamNames()
    for name, label in zip(name_list, label_list):
        param_names.parWithName(name).label = label

In [ ]:
from astropy.cosmology import Planck18 as planck

Omega_m_fid = planck.Om0
sigma_8_fid = 0.8054
s8_fid = sigma_8_fid * (Omega_m_fid / 0.3) ** 0.5
h = planck.h
Omega_b_fig = planck.Ob0
n_s_fid = 0.965
print(
    f"Fiducial values: Omega_m = {Omega_m_fid}, sigma_8 = {sigma_8_fid}, S_8 = {s8_fid}"
)

markers = {
    "OMEGA_M": Omega_m_fid,
    "SIGMA_8": sigma_8_fid,
    "s_8_input": s8_fid,
    "h0": h,
    "ombh2": Omega_b_fig * h**2,
    "n_s": n_s_fid,
}

In [ ]:
legend_labels = [rf"$\xi_\pm(\vartheta)$, Mock {i}" for i in range(1, 3)] + [
    rf"$C_\ell$, Mock {i}" for i in range(1, 3)
]

# Plot all parameters
g.triangle_plot(
    chains,
    [
        "OMEGA_M",
        "ombh2",
        "h0",
        "n_s",
        "SIGMA_8",
        "s_8_input",
        "logt_agn",
        "a",
        "m1",
        "bias_1",
    ],
    legend_labels=legend_labels,
    legend_loc="upper right",
    filled=True,
    markers=markers,
)

g.export("plots/contours_all_cell_glass_mock.png")
plt.show()

In [ ]:
# Plot only cosmological parameters

g.triangle_plot(
    chains,
    ["OMEGA_M", "s_8_input", "SIGMA_8", "a"],
    legend_labels=legend_labels,
    legend_loc="upper right",
    filled=True,
    markers=markers,
)

g.export("plots/contours_cosmo_cell_glass_mock.png")
plt.show()

In [ ]:
# Plot S8 Omega_m only
g.triangle_plot(
    chains,
    ["OMEGA_M", "s_8_input"],
    legend_labels=legend_labels,
    legend_loc="upper right",
    filled=True,
    title_limit=1,
    markers=markers,
)

""" plt.figtext(0.5, 0.5, 'PRELIMINARY',
            fontsize=150, color='gray',
            ha='center', va='center',
            alpha=0.3, rotation=330) """

g.export("plots/contours_s8_omegam_cell_glass_mock.png")
plt.show()

In [ ]:
s8_values = np.array(["# Expt", "Mean", "S8_low", "S8_high"])
for i, chain in enumerate(chains):
    margestats = chain.getMargeStats()
    likestats = chain.getLikeStats()

    param_stats = margestats.parWithName("S_8")

    s8_values = np.vstack(
        (
            s8_values,
            [
                roots[i],
                param_stats.mean,
                param_stats.mean - param_stats.limits[0].lower,
                param_stats.limits[0].upper - param_stats.mean,
            ],
        )
    )
print(s8_values)
np.savetxt(
    f"{root_dir}/S8_means.txt", s8_values, fmt=["%s", "%s", "%s", "%s"], delimiter=","
)

In [ ]:
s8s = np.loadtxt(
    f"{root_dir}/S8_means.txt",
    dtype={
        "names": ("Expt", "s8_mean", "s8_low", "s8_high"),
        "formats": ("U40", "U20", "U20", "U20"),
    },
    skiprows=1,
    delimiter=",",
)
expt = s8s["Expt"]
s8s_mean = s8s["s8_mean"].astype(np.float64)
s8s_low = s8s["s8_low"].astype(np.float64)
s8s_high = s8s["s8_high"].astype(np.float64)

In [ ]:
num_test_cases = 1

fig, axs = plt.subplots(1, 1, sharey=True, figsize=[7, 1.5 * len(expt)])
axs.yaxis.set_visible(False)

y = np.arange(0, len(expt))

for i in y:
    if i > len(expt) - num_test_cases - 1:
        axs.errorbar(
            s8s_mean[i],
            i + 1,
            xerr=np.vstack((s8s_low[i], s8s_high[i])),
            fmt="o",
            c="darkblue",
            lw=2,
            capsize=5,
            capthick=2,
        )
    else:
        axs.errorbar(
            s8s_mean[i],
            i + 1,
            xerr=np.vstack((s8s_low[i], s8s_high[i])),
            fmt="o",
            c="darkgreen",
            lw=2,
            capsize=5,
            capthick=2,
        )

    """ if i == 0:        # Plot the band for "Planck"
        axs.axvspan(s8s_mean[i]-s8s_low[i], s8s_mean[i]+s8s_high[i], alpha=0.2, color='cyan')
    if i == len(expt)-1:        # Plot the band for "this work"
        axs.axvspan(s8s_mean[i]-s8s_low[i], s8s_mean[i]+s8s_high[i], alpha=0.2, color='lightpink')"""
    if (
        i == len(expt) - num_test_cases - 1
    ):  # Make a distinction between this work (and all its test cases) with external datasets
        axs.axhline(i + 1.5, ls="dashed", c="k")

    axs.set_xlabel(r"$S_8=\sigma_8\sqrt{\Omega_{\rm m}/0.3}$")
    axs.text(0.62, i + 1, rf"{expt[i]}")

plt.ylim([0, i + 2])
plt.savefig("./plots/S8_whisker.png")